# NutriFit 전체 상품 데이터 탐색적 분석 (EDA)

이 보고서는 NutriFit 전체 상품 수집 데이터(`NutriFit-Dashboard-Data-file.csv`, 전체 27,779건)를 분석한 보고서입니다.

### 1. 플랫폼별(Coupang/OliveYoung/iHerb) 건수 및 주요 컬럼 결측치 비율(%)
**이 분석을 왜 하는지**: 전체 데이터셋에서 수집된 플랫폼별 데이터의 분포와 각 컬럼의 누락율(결측치 비율)을 파악하여 데이터의 품질을 검증하고 전처리 방향을 설정하기 위함입니다.

In [1]:
import pandas as pd
import numpy as np

# 데이터 파일 로드
df_csv = pd.read_csv('NutriFit-Dashboard-Data-file.csv', low_memory=False)

print('=== 플랫폼별 전체 데이터 건수 ===')
print(df_csv['platform'].value_counts().to_string())

print('\n=== 플랫폼별 주요 컬럼 결측치 비율 (%) ===')
for platform, group in df_csv.groupby('platform'):
    print(f'\n[ {platform} ] (총 {len(group)}건)')
    null_ratios = group.isnull().mean() * 100
    active_nulls = null_ratios[null_ratios < 100].round(2)
    print(active_nulls.to_string())

=== 플랫폼별 전체 데이터 건수 ===
platform
iHerb         24711
OliveYoung     2560
Coupang         508

=== 플랫폼별 주요 컬럼 결측치 비율 (%) ===

[ Coupang ] (총 508건)
product_id         0.00
product_name       0.00
price              0.00
unit_price         0.98
rocket_delivery    0.00
rating             0.00
review_count       0.00
image_url          0.00
detail_url         0.00
collected_at       0.00
platform           0.00

[ OliveYoung ] (총 2560건)
review_count     9.57
카테고리             0.00
goods_no         0.00
brand            0.00
name             0.00
price_org       40.43
price_cur        0.00
score            9.57
tags            14.22
link             0.00
img_url          0.00
platform         0.00

[ iHerb ] (총 24711건)
rating             0.00
productId          0.00
displayName        0.00
brandName          0.00
partNumber         0.00
listPrice          0.00
discountPrice      0.00
ratingCount        0.00
url                0.00
packageQuantity    0.08
productForm        1.00
pricePerServing

### 2. 플랫폼별 가격 컬럼 기초 통계(최소/최대/평균/중앙값)
**이 분석을 왜 하는지**: 각 플랫폼별 제품의 가격 범주(최소값, 최대값, 평균, 중앙값)를 계산하여 가격대의 분포와 특징을 비교 분석하기 위함입니다.

In [2]:
# 가격 정제 함수 정의
def clean_price_val(val):
    if pd.isna(val):
        return np.nan
    val_str = str(val).strip()
    if '.' in val_str:
        val_str = val_str.split('.')[0]
    cleaned = ''.join([c for c in val_str if c.isdigit()])
    return float(cleaned) if cleaned else np.nan

print('=== 플랫폼별 가격 기초 통계량 ===')
for platform in ['Coupang', 'OliveYoung', 'iHerb']:
    group = df_csv[df_csv['platform'] == platform]
    if platform == 'Coupang':
        price_col = 'price'
    elif platform == 'OliveYoung':
        price_col = 'price_cur'
    else:
        price_col = 'discountPrice'
        
    prices = group[price_col].apply(clean_price_val).dropna()
    print(f'\n[ {platform} ] ({price_col} 컬럼 기준)')
    print(f'  - 최소값: {prices.min():,.0f}원')
    print(f'  - 최대값: {prices.max():,.0f}원')
    print(f'  - 평균값: {prices.mean():,.2f}원')
    print(f'  - 중앙값: {prices.median():,.0f}원')

=== 플랫폼별 가격 기초 통계량 ===

[ Coupang ] (price 컬럼 기준)
  - 최소값: 2,500원
  - 최대값: 192,690원
  - 평균값: 27,639.02원
  - 중앙값: 20,950원

[ OliveYoung ] (price_cur 컬럼 기준)
  - 최소값: 1,400원
  - 최대값: 784,000원
  - 평균값: 31,833.47원
  - 중앙값: 23,900원

[ iHerb ] (discountPrice 컬럼 기준)
  - 최소값: 0원
  - 최대값: 322,298원
  - 평균값: 37,007.17원
  - 중앙값: 29,803원

